In [ ]:
import matplotlib.pyplot as plt
import netCDF4 as nc4
import xarray as xr
import numpy as np

import os

#from microhhpy.io import read_ini

In [ ]:
"""
Help functions.
"""

def xr_read_all(f, groups=['default'], decode_times=False):
    """
    Read NetCDF with Xarray including groups.
    """
    dss = [xr.open_dataset(f, decode_times=decode_times)]
    for group in groups:
        dss.append(xr.open_dataset(f, group=group, decode_times=decode_times))
    ds = xr.merge(dss)

    fill_value = nc4.default_fillvals['f4']
    for var in ds.data_vars:
        ds[var] = ds[var].where(ds[var] != fill_value)

    return ds


def add_sub_axes(x0, y0, width, ax, xlim=None, x00=None):
    """
    Add a secondary axes at data coordinates {x0, y0} of `ax`,
    with only a top x-axis and the same y-extent as `ax`.
    """
    fig = ax.figure

    # Data -> pixel -> figure (0..1) coordinates.
    t = ax.transData.transform((x0, y0))
    t = fig.transFigure.inverted().transform(t)

    if xlim is not None and x00 is not None:
        xoffs = -(x00-xlim[0])/(xlim[1]-xlim[0])*width
    else:
        xoffs = 0

    new_ax = fig.add_axes([t[0]+xoffs, t[1], width, ax.get_position().height])
    new_ax.set_facecolor('none')
    new_ax.set_ylim(ax.get_ylim())
    new_ax.yaxis.set_ticks([])

    new_ax.spines['right'].set_visible(False)
    new_ax.spines['left'].set_visible(False)
    new_ax.spines['bottom'].set_visible(False)
    new_ax.spines['top'].set_visible(True)
    new_ax.get_xaxis().tick_top()
    new_ax.xaxis.set_label_position('top')

    if xlim is not None:
        new_ax.set_xlim(xlim)

    new_ax.axvline(x=0, color='gray', linestyle=':', linewidth=0.5)

    return new_ax

In [ ]:
"""
Case settings.
"""
amplitude = 0.00254
xsize = 0.1016
wavelength = 0.1016
z_offset = 0.002
h = 5.08 / 100 / 2.

def calc_z_ib(x, norm=True):
    z = amplitude + amplitude * np.cos(2*np.pi*x/wavelength)

    if norm:
        z = (z + z_offset) / amplitude

    return z

x = np.linspace(0, xsize, 256)
xn = x / wavelength
z_ib = calc_z_ib(x, norm=False)

In [ ]:
class Read_hudson:
    """
    Read the data files from Hudson (1993)
    """
    def __init__(self, path, loc):

        fu = np.loadtxt(os.path.join(path, f'xlam{loc:02d}_u.txt'), skiprows=4)
        fv = np.loadtxt(os.path.join(path, f'xlam{loc:02d}_v.txt'), skiprows=4)
        ft = np.loadtxt(os.path.join(path, f'xlam{loc:02d}_t.txt'), skiprows=4)

        fac = 0.01 * np.sqrt(2)  # cm -> m, and correction as mentioned in readme.txt
        self.z  = fu[:,0] * fac
        self.u  = fu[:,1] * fac
        self.v  = fv[:,1] * fac
        self.u2 = ft[:,0] * fac
        self.v2 = ft[:,1] * fac
        self.uw = ft[:,2] * fac * 0.01

        # Hard-coded settings............. (-.-)
        self.ustar = 1.29 / 100         # See readme.txt
        self.wave_length = 5.08 / 100.
        self.h = self.wave_length / 2.  # Half depth channel = half wave length
        self.amplitude = 0.05 * self.wave_length

        # Location and height obs as reported in paper
        self.x_norm = np.arange(0,1,0.1)   # x/lambda
        self.z_norm = np.array([0.2, 0.178, 0.132, 0.072, 0.019, 0.003, 0.039, 0.101, 0.157, 0.190])    # (z - z0) / h

        #self.z += self.z_norm[loc]*self.h

        # Nearest location on model boundary
        self.x_near = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.645, 0.75, 0.85, 0.933])
        #self.x_near = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.62, 0.73, 0.82, 0.91])
        #self.x_near = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

#hudson = Read_hudson('reference_data/Hudson_1993', 0)
hudson = Read_hudson('reference_data/', 0)

In [ ]:
"""
Read DNS experiments.
"""
#base_path = '/home/scratch2/bart/microhh/microhh2_paper/ib_sine'
#base_path = '/home/bart/meteo/models/microhh/cases/ib_sine'
base_path = '.'

class Read_exp:
    def __init__(self, exp, t0, t1):

        # Domain mean statistics, excluding IB.
        self.stat = xr_read_all(f'{base_path}/{exp}/sine.ib.0000000.nc')


        self.stat_m = self.stat.sel(time=slice(t0, t1))

        # Spanwise slices at observation locations Hudson.
        self.u_yz = xr.open_dataset(f'{base_path}/{exp}/u.yz.nc', decode_times=False)

        self.u_yz = self.u_yz.sel(time=slice(t0, t1))
        self.u_yz_m = self.u_yz.mean(dim=('time', 'y'))

        # Variance profiles `u`.
        self.u_yz_span = self.u_yz.mean(dim='y')
        self.u_yz_p = (self.u_yz - self.u_yz_span)**2
        self.u_yz_pm = self.u_yz_p.mean(dim=('time', 'y'))

        # Read .ini file.
        ini = read_ini(f'{base_path}/{exp}/sine.ini')

        # Calculate statistics.
        self.z_size = float(self.stat.zh[-1])
        self.z_eff = self.z_size - z_offset

        self.dpdx = ini['force']['dpdx']
        self.visc = ini['fields']['visc']

        self.ustar = np.sqrt(np.abs(self.dpdx) * self.z_eff)
        self.u_mean = float(self.stat_m.u.mean())
        self.Re = self.u_mean * h / self.visc

        self.z_norm = (self.stat.z - amplitude - z_offset) / h

        print(f'u_mean = {self.u_mean:.3e}, u* = {self.ustar:.3e}, Re={self.Re:.0f}')

        # Plot for steady-state bulk velocity.
        u_mean = self.stat.u.mean(dim='z')  # TMP
        
        plt.figure()
        plt.title(f'{exp}: {self.u_mean:.3f}')
        plt.plot(self.stat.time, u_mean)


t0 = 0
t1 = 100
r1 = Read_exp('', t0, t1)

exps = [r1]

#t0 = 900
#t1 = 1200
#r1 = Read_exp('v2_256_128_128_6.5e5_0.00137', t0, t1)
#r2 = Read_exp('v2_256_128_128_13e5_0.00137', t0, t1)
#r4 = Read_exp('v2_512_256_256_50e5_0.00137', t0, t1)
#exps = [r1, r2, r4]

In [ ]:
"""
Validation: mean and variance profiles.
"""
# Times to analyze
t0 = 200
t1 = 300

# Plot settings
mc_ref = 'k'
ms_ref = 5

c_ib = '0.9'


fig=plt.figure(figsize=(8, 6))
fig.subplots_adjust(left=0.06, bottom=0.06, right=0.98, top=0.93, hspace=0.25)

def setup_ax(n):
    ax=plt.subplot(2,1,n)
    ax.plot(xn, (z_ib - amplitude)/h, 'k-', linewidth=0.75)
    ax.fill_between(xn, -0.1, (z_ib - amplitude)/h, color=c_ib, alpha=1.0)
    ax.set_ylim(-0.1, 0.6)
    ax.set_xlim(0, 1)
    ax.set_xticks(np.arange(0,1.01,0.1))
    return ax

ax_m1 = setup_ax(1)
ax_m2 = setup_ax(2)

# Mean profiles
for i in range(0,10,1):
    x_hat = hudson.x_near[i]

    x0 = wavelength / 4. + x_hat * wavelength
    z0 = amplitude * np.sin(2*np.pi*(x0)/wavelength)

    hudson = Read_hudson('reference_data/Hudson_1993', i)
   
    ax = add_sub_axes(x_hat, -0.1, 0.08, ax=ax_m1, xlim=[-2,14], x00=0)
    ax.plot(hudson.u/hudson.ustar, (hudson.z+z0)/h, 'o', ms=ms_ref, mfc='none', mec=mc_ref, alpha=1.0, label='Hudson (1993)')
    for exp in exps:
        ax.plot(exp.u_yz_m.u[:,i] / exp.ustar, exp.z_norm, label=f'Re={exp.Re:.0f}')
    ax.set_xlabel(r'$\widehat{u}$')
    ax.set_xticks([0,4,8,12])

    if (i == 0):
        ax.legend(frameon=False, loc='lower left') 

# Variance profiles
for i in range(0,10,1):
    x_hat = hudson.x_near[i]

    x0 = wavelength / 4. + x_hat * wavelength
    z0 = amplitude * np.sin(2*np.pi*(x0)/wavelength)
    
    hudson = Read_hudson('reference_data/Hudson_1993', i)
   
    ax = add_sub_axes(x_hat, -0.1, 0.07, ax=ax_m2, xlim=[0,3], x00=0)
    ax.set_xticks([0,1,2,3])
    ax.plot(hudson.u2/hudson.ustar, (hudson.z+z0)/h, 'o', ms=ms_ref, mfc='none', mec=mc_ref, alpha=1.0)
    for exp in exps:
        ax.plot(np.sqrt(exp.u_yz_pm.u[:,i]) / exp.ustar, exp.z_norm)
    
    ax.set_xlabel(r'$\widehat{u}^{\prime 2}$')

ax_m1.set_ylabel(r'$\widehat{z}$')
ax_m2.set_ylabel(r'$\widehat{z}$')
ax_m1.set_xticklabels([])
ax_m2.set_xlabel(r'$\widehat{x}$')